# Canadian Airlines Delay Classification

Build a complete preprocessing and logistic-regression pipeline using synthetic Canadian airline data.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 1200
airports = ["YYZ", "YVR", "YUL", "YYC", "YOW"]
airlines = ["Air Canada", "WestJet", "Porter"]

df = pd.DataFrame({
    "origin": rng.choice(airports, n),
    "destination": rng.choice(airports, n),
    "airline": rng.choice(airlines, n),
    "distance_km": rng.integers(300, 4200, n),
    "scheduled_hour": rng.integers(5, 23, n),
    "weather_severity": rng.integers(0, 4, n),
})
logit = -2.2 + 0.00025*df["distance_km"] + 0.38*df["weather_severity"] + 0.04*(df["scheduled_hour"]-12).clip(lower=0)
prob = 1/(1+np.exp(-logit))
df["delayed_15min"] = rng.binomial(1, prob)
df.head()


In [ ]:
X = df.drop(columns="delayed_15min")
y = df["delayed_15min"]

categorical = ["origin", "destination", "airline"]
numeric = ["distance_km", "scheduled_hour", "weather_severity"]

preprocess = ColumnTransformer([
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical),
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=2000))
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "roc_auc": roc_auc_score(y_test, prob),
    "classification_report": classification_report(y_test, pred, output_dict=True)
}
metrics["accuracy"], metrics["roc_auc"]


In [ ]:
sample = X_test.copy()
sample["actual"] = y_test
sample["predicted"] = pred
sample["delay_probability"] = prob
sample.head(50).to_csv("prediction_sample.csv", index=False)

with open("airline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
